# Rod-bundle PINN — step-by-step walkthrough

Trains the physics-informed neural network on your OpenFOAM data, tests how much data it
needs, explains what it learned, and checks it against the 2023 DNS paper.
Kernel: pick **enygf**. It uses the same code as `src/`, so results match a terminal run.

**How to run everything:** Kernel → Restart, then **Run → Run All Cells**. Every step can
be re-run safely: finished trainings are saved and skipped, so an interrupted run just
continues where it stopped.

| Step | What | Time on a laptop CPU |
|---|---|---|
| 1 | Look at the CFD data | seconds |
| 2–5 | Train the full-data PINN (skipped if already trained) and evaluate it | 0 min if trained, else 30–60 min |
| 6 | Sparse-data experiment: 100% / 10% / 1% / lines, PINN vs plain NN | **the long one: ~5–7 h** |
| 7 | Explainability: residual maps, energy budget | ~1 min |
| 8 | Sensor importance (which measurement line matters) | ~2–4 h |

Start Run All in the evening and keep the laptop plugged in and awake.

Needed in `digitized_data/cfd_generated/` (written by `cfd/openfoam/extract_profiles.py`):
`cfd_field.csv`, `cfd_nusselt.csv`, `cfd_profiles.csv`.

In [ ]:
import os, sys
from pathlib import Path

# Work from ml/ so the paths in configs/default.yaml resolve
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd() / "src"))

import yaml, torch, pandas as pd, matplotlib.pyplot as plt
import train as trainer
import evaluate as evaluator

cfg = yaml.safe_load(open("configs/default.yaml"))
print("working dir:", Path.cwd())
print("python:", sys.executable)
print("device:", "GPU (CUDA)" if torch.cuda.is_available() else "CPU")

## Step 1 — Look at the training data

The CFD profiles along the path the DNS paper plots on (`xi`): rod surface in the narrow
gap → gap centre → subchannel centre → back to the rod at 45°. Then the CFD's own Nusselt
numbers next to the DNS — this shows how far the RANS model itself is from the DNS,
before any machine learning.

In [ ]:
df = pd.read_csv(cfg["paths"]["cfd_profiles"])
field = pd.read_csv(cfg["paths"]["cfd_field"])
print(field.groupby(["kind", "quantity"]).size().rename("training points"))

path = df[df["line"].isin(["seg1", "seg2", "seg3"])]
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, q, label in zip(axes, ["w", "nut", "k"], ["axial velocity w [m/s]", "eddy viscosity nu_t [m2/s]", "TKE k [m2/s2]"]):
    g = path[path["quantity"] == q].sort_values("xi")
    ax.plot(g["xi"] / cfg["geometry"]["dh_dns"], g["value"], ".", ms=2)
    ax.set_xlabel("xi / Dh"); ax.set_title(label)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, bc in zip(axes, ["isoT", "isoFlux"]):
    for case, g in path[(path["quantity"] == "T") & (path["bc"] == bc)].groupby("case_id"):
        g = g.sort_values("xi")
        ax.plot(g["xi"] / cfg["geometry"]["dh_dns"], g["value"], ".", ms=2, label=case)
    ax.set_xlabel("xi / Dh"); ax.set_title(f"temperature, {bc}"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

nu_cfd = pd.read_csv(cfg["paths"]["cfd_nusselt"])
dns = pd.read_csv(cfg["paths"]["dns_nusselt"]).melt(id_vars="Pr", value_vars=["Nu_isoT", "Nu_isoFlux"], var_name="bc", value_name="Nu_DNS")
dns["bc"] = dns["bc"].str.replace("Nu_", "")
t = nu_cfd.merge(dns, on=["Pr", "bc"], how="left")
t["CFD_vs_DNS"] = (t["Nu_CFD"] / t["Nu_DNS"] - 1).map(lambda v: f"{v:+.0%}" if v == v else "-")
t["Pr"] = t["Pr"].map("{:g}".format)
print("\nCFD Nusselt numbers vs. DNS (both with Dh = 0.0712 m):")
print(t[["Pr", "bc", "Nu_CFD", "Nu_DNS", "CFD_vs_DNS"]].to_string(index=False, float_format=lambda v: f"{v:.2f}"))

## Step 2 — Settings

- `QUICK = True` shrinks every training to 100 epochs to check that all cells run (~15 min
  for the whole notebook). Results are meaningless and saved separately.
- `RETRAIN = False` reuses the full-data PINN you already trained (`outputs/pinn_checkpoint.pt`);
  set `True` to train it again.
- `INCLUDE_SEEDS = True` repeats the 10%, 1%, lines-only and sensor-importance runs with 2
  more random seeds (adds several hours) — needed for the paper: with little data the result
  depends on which cells were picked and how the network started, and the seed spread shows
  which conclusions are robust.

In [ ]:
QUICK = False
RETRAIN = False
INCLUDE_SEEDS = True

if QUICK:
    cfg["training"].update(epochs_data=50, epochs_physics=50, ramp_epochs=20, print_every=10)
    cfg["paths"].update(checkpoint="outputs_quick/pinn_checkpoint.pt",
                        loss_history="outputs_quick/loss_history.csv", plots_dir="outputs_quick/plots")
EXP_DIR = "outputs_quick/experiments" if QUICK else "outputs/experiments"
have_model = Path(cfg["paths"]["checkpoint"]).exists()
print("epochs per training:", cfg["training"]["epochs_data"] + cfg["training"]["epochs_physics"])
print("full-data PINN:", "will be REUSED from " + cfg["paths"]["checkpoint"] if have_model and not RETRAIN else "will be TRAINED")

## Step 3 — Train

- **Phase 1** (`epochs_data`): the network fits the CFD data — velocity, eddy viscosity, TKE,
  all 8 temperature fields, wall shear and wall heat flux — while keeping the bulk velocity
  at 1 m/s and zero gradients on the symmetry planes.
- **Phase 2** (`epochs_physics`): the governing equations (axial momentum, energy), the
  iso-flux wall condition and the overall force balance are switched on gradually, so the
  fields become physically consistent (Step 6 measures what this adds).

20% of the CFD cells are **held out**: the network never trains on them, and the error on
them at the end is the honest measure of accuracy.

In [ ]:
if have_model and not RETRAIN:
    print("Using the already-trained model:", cfg["paths"]["checkpoint"], "(set RETRAIN = True in Step 2 to retrain)")
else:
    model = trainer.train(cfg)

## Step 4 — Loss curves

What to look for:
- **`data_*` curves fall** in phase 1 and stay low in phase 2.
- **Physics terms appear at the dotted line** (`pde_*`, `wall_heat_flux`, `wall_shear_balance`)
  and fall from there. They stay higher than the data terms; that is normal.
- **`dpdz`** (learned pressure gradient) jumps when physics switches on and should settle on
  the CFD value (black dashed line, computed from the CFD wall shear). The red dotted line is
  the DNS value — the gap between the two lines is a RANS-vs-DNS difference, not a PINN error.

In [ ]:
hist = pd.read_csv(cfg["paths"]["loss_history"])
terms = [c for c in hist.columns if c not in ("epoch", "phase", "total", "dpdz")]
switch = cfg["training"]["epochs_data"]

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for t in terms:
    s = hist[["epoch", t]].dropna()
    axes[0].semilogy(s["epoch"], s[t].rolling(20, min_periods=1).mean(), label=t)
axes[0].axvline(switch, color="gray", ls=":")
axes[0].set_xlabel("epoch"); axes[0].set_title("loss terms (20-epoch moving average)")
axes[0].legend(fontsize=7, ncol=2)

# CFD pressure gradient from its wall shear: G * area = sum(tau_w) * rod arc length
import math
g = cfg["geometry"]
tau = field[field["quantity"] == "tau_w"]["value"].mean()
G_cfd = tau * (math.pi * g["rod_radius"] / 2) / (g["half_pitch"] ** 2 - math.pi * g["rod_radius"] ** 2 / 4)
axes[1].plot(hist["epoch"], hist["dpdz"], label="PINN")
axes[1].axhline(G_cfd, color="k", ls="--", label=f"OpenFOAM CFD ({G_cfd:.3f})")
axes[1].axhline(4 * 0.0637**2 / g["dh_cell"], color="r", ls=":", label="from DNS u_tau (0.207)")
axes[1].set_xlabel("epoch"); axes[1].set_title("learned pressure gradient dp/dz [m/s2]"); axes[1].legend()
plt.tight_layout(); plt.show()

## Step 5 — Evaluate

1. **Held-out CFD cells** — error on points the network never saw (`rmse_rel` = error / range
   of that field; below ~0.02 is a good fit).
2. **Along the sampled lines** — profile plots, PINN vs. CFD.
3. **Nusselt numbers** — PINN vs. the CFD it learned from vs. the DNS (Table 1 of the 2023 paper).
   `PINN_vs_CFD` measures the machine learning; `CFD_vs_DNS` measures the RANS model.
   The DNS was never used in training.
4. **Digitized DNS figures**, if you have filled in `digitized_data/dns_*.csv`.

In [ ]:
fit, nusselt, dns = evaluator.evaluate(cfg)

In [ ]:
from IPython.display import Image, display
for png in sorted(Path(cfg["paths"]["plots_dir"]).glob("*.png")):
    print(png.name)
    display(Image(filename=str(png)))

## Step 6 — Sparse-data experiment (the research question)

How much CFD data does the network need, and what do the physics terms add? The same
network is trained **with** the equations (PINN) and **without** them (plain NN) on shrinking
amounts of data, and every run is tested on the **same held-out cells** and against the CFD
Nusselt numbers (`docs/research_plan.md`).

| Row | Training data |
|---|---|
| 100% | every training cell + wall data (the 100% PINN is your Step 3 model, not retrained) |
| 10% / 1% | a random 10% / 1% of the cells and of the wall points |
| lines | only the 4 sampled lines, no wall data — the most sensor-like case |

Output: a table, `outputs/experiments/summary.csv`, and the bar chart
`outputs/experiments/summary.png` (log scale; the whiskers on the 1% bars show the seed spread).
What to look for: with lots of data the plain NN may fit slightly better; the question is
whether the PINN stays accurate as the data shrinks.

In [ ]:
import experiments as ex

runs = dict(ex.RUNS)
runs["100% PINN"] = {"checkpoint": cfg["paths"]["checkpoint"]}   # reuse the Step 3 model
if INCLUDE_SEEDS:
    runs.update(ex.SEED_RUNS)
table = ex.run_all(cfg, runs, out_dir=EXP_DIR, plot=True)

## Step 7 — Explainability: what did the PINN learn, and where can it be trusted?

Standard XAI tools (SHAP, LIME) rank input features; this network's inputs are only the
coordinates, so they would not explain anything physical. Instead, two physics-based views of
the full-data PINN:

1. **Residual maps** — how badly the network breaks the momentum and energy equations at each
   point (log scale: −2 = 1% of the driving term, 0 = 100%). Dark = trustworthy, bright = not.
   The table gives the RMS by region: near the wall, in the narrow gap, at the subchannel centre.
2. **Energy budget** — what share of the heat is carried by turbulent mixing rather than
   molecular conduction, from the network's own eddy viscosity. This is the model explaining its
   Nusselt numbers: where turbulence carries the heat, the RANS turbulence errors matter.
   Expect: liquid metal (Pr = 0.025) mostly conduction, especially in the gap; Pr ≥ 1 mostly
   turbulence except in a thin wall layer.

In [ ]:
import xai

model = xai.load_trained(cfg)
residual_table = xai.residual_maps(model, cfg)
budget_table = xai.energy_budget(model, cfg)

## Step 8 — Sensor importance: where are measurements most valuable?

The lines-only PINN from Step 6 is retrained 4 times, each time **without one** of the four
measurement lines (seg1 = narrow gap, seg2 = gap-to-subchannel centreline, seg3 = subchannel
centre to rod at 45°, line15 = wall-normal line at 15°). The more the error grows when a line
is removed, the more that line matters. For a reactor, this is the question "where should the
thermocouples go?".

In [ ]:
sensor_runs = {"lines PINN": ex.RUNS["lines PINN"], **ex.SENSOR_RUNS}
if INCLUDE_SEEDS:
    sensor_runs.update(ex.SENSOR_SEED_RUNS)     # the seed-1/2 references come from Step 6
sensor_table = ex.run_all(cfg, sensor_runs, out_dir=EXP_DIR)

import json
results = [json.loads(p.read_text()) for p in Path(EXP_DIR).glob("*/result.json")]
importance = xai.sensor_importance(results, out=Path(EXP_DIR) / "sensor_importance.png")

## If it doesn't converge well

Change one thing at a time in `configs/default.yaml` (or in `cfg` above) and rerun from Step 2:

- **Losses still falling at the end** → train longer: increase `epochs_physics`.
- **Curves noisy or jumping** → lower `lr` (e.g. `5.0e-4`).
- **Held-out error good but `PINN_vs_CFD` Nusselt off for iso-flux** → raise `wall_heat_flux`
  in `loss_weights`.
- **`dpdz` not settling on the CFD value** → raise `wall_shear_balance`.

Write down each change and its effect. That log is part of your methodology section.